# Phase 1 Chunk 3

# Explorations


# ______________________________________________________________________

### Original code from Basics 03 :

In [1]:
import torch
from sklearn.datasets import make_regression

torch.manual_seed(42)


# 1. Loading the dataset and define parameters

# 100 houses and 1 feature i:e sq.footage

X,y = make_regression(n_samples=100,
                      n_features=1,
                      noise=10,
                      random_state=42)

# 2. Convert to tensors


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# NOTE : y needs to be reshaped to [100, 1] to match the prediction shape
X_tensor = torch.tensor(X, dtype=torch.float32, device = device)
y_tensor = torch.tensor(y, dtype=torch.float32, device=device).view(100,1)


# 3. The parameterrs we want the model to learn are w and b.
# So we must tell Pytorch that we want gradients for them.
w = torch.randn(1,1, dtype=torch.float32, device=device, requires_grad=True) # A random weight
b = torch.randn(1, dtype=torch.float32, device=device, requires_grad=True)   # A random bias


print(f"Initial w : {w.item():.4f},\nInitial b : {b.item():.4f}")


# 4. The Forward pass
# The same as before
y_pred = X_tensor @ w + b  # Matrix multiplication

# 5. Calculate loss
# We will use MSE. A standard for regression
loss1 = torch.nn.functional.mse_loss(y_pred, y_tensor)
# OR
loss2 = torch.mean((y_pred - y_tensor)**2)

print(f"Loss 1 and loss 2 are equal : {loss1 == loss2}")
print(f"Initial loss : {loss1.item():.4f}")

# 6. The Backward pass
# Pytorch traces the computations backward from `loss` to `w` to `b`
# and calculates the gradients automatically

loss1.backward()

# 7. Inspect the gradients
w_grad = w.grad
b_grad = b.grad
print("\n   After backward pass   ")
print(f"Gradient w : {w_grad.item():.4f},\nGradient b : {b_grad.item():.4f}")


print("\nWhat does w.grad mean?")
print("It means if we increase 'w' by a tiny amount," \
"\nthe loss will increase by 'w.grad' times that amount." \
"\nTo DECREASE the loss," \
"\nwe need to move in the opposite direction of the gradient")


Initial w : 0.1940,
Initial b : 0.1391
Loss 1 and loss 2 are equal : True
Initial loss : 1689.0951

   After backward pass   
Gradient w : -72.9922,
Gradient b : 7.1370

What does w.grad mean?
It means if we increase 'w' by a tiny amount,
the loss will increase by 'w.grad' times that amount.
To DECREASE the loss,
we need to move in the opposite direction of the gradient


# ______________________________________________________________________

### Intermediate

In [2]:
w.grad
# loss1.backward() # This accumulates the gradients


tensor([[-72.9922]], device='cuda:0')

Calling loss. backward() a secod time causes an error

### Advanced

In [3]:
w = torch.randn(1,1, dtype=torch.float32, device=device, requires_grad=True) # A random weight
b = torch.randn(1, dtype=torch.float32, device=device, requires_grad=True)   # A random bias


print(f"Initial w : {w.item():.4f},\nInitial b : {b.item():.4f}")


# 4. The Forward pass
# The same as before
y_preds = X_tensor @ w + b  # Matrix multiplication
loss = torch.mean(((y_preds - y_tensor) ** 2))
loss



Initial w : -0.5187,
Initial b : -0.6974


tensor(1736.1475, device='cuda:0', grad_fn=<MeanBackward0>)

In [4]:
# dL/dw = d_loss / dy_preds * dy_preds / d_weights

#### For our model

# y_preds = X*w + b
# dy_preds / d_weights = X


# Since We are using MSE loss
# loss = (y_preds - y_true)^2
# d_loss / dy_preds = 2 * (y_preds - y_true)

# Calculating DL/dw Manually

DL_DW = torch.mean((2 * (y_preds - y_tensor) * X_tensor)) * X_tensor
DL_DW.retain_grad()

In [5]:
print(w.grad)

None


In [6]:
DL_DW[0] -w

tensor([[-68.3939]], device='cuda:0', grad_fn=<SubBackward0>)

#### Close...... Because the weight Initialization of second time is different therefore there might be a difference. Anyways We did it. We calculated backpropagation manually

# 5. Dataset-Specific Excercises

## Replication

In [7]:
import torch
from sklearn.datasets import _california_housing

X, y = _california_housing.fetch_california_housing(return_X_y=True,
                                                    as_frame=True)
import pandas as pd
print(f"Shape of X and y Respectively : {X.shape}, {y.shape}")
print(f"Feature names of the dataset : {X.columns.tolist()},\nTraget name {pd.DataFrame(y).columns.tolist()}")
print(f"\nWe need the feature ' AveRooms ' as X \nand\ntarget ' MedHouseVal ' as y")
print(f"\nThen we need to compute one forward and backward pass\nOn Our X and y")


df = pd.concat([X, y], axis=1)
df.head(3)

df_for_exploration = df[['AveRooms', 'MedHouseVal']]
df_for_exploration.head(3)
X_exp = df_for_exploration['AveRooms']
y_exp = df_for_exploration['MedHouseVal']



# X_exp and y_exp are in dataframe dtype. They need to be converted to numpy and then tensors
import numpy as np
X_np, y_np = np.array(X_exp), np.array(y_exp)
X_np = X_np.reshape(-1,1)  # Reshaping to [20640, 1]
y_np = y_np.reshape(-1,1)  # Reshaping to [20640, 1]

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X_tensor = torch.tensor(X_np,
                        dtype=torch.float32, 
                        device=device,
                        requires_grad=True)
y_tensor = torch.tensor(y_np,
                        dtype=torch.float32,
                        device=device,
                        requires_grad=True)

w_exp = torch.randn(1,1, device=device, requires_grad=True)
b_exp = torch.randn(1,1, device=device, requires_grad=True)
torch.randn(1, device=device)
y_preds_exp = X_tensor @ w_exp + b_exp
print(f"Initial Weight and Bias values resprctively : {w_exp.item():.4f},\n{b_exp.item():.4f}")
loss_exp = torch.mean((y_preds_exp - y_tensor)**2)


print(f"Initial loss: {loss_exp.item()}")




loss_exp.backward()
w_grad_exp = w_exp.grad
b_grad_exp = b_exp.grad
print(f"Gradient of w and b are respectively : {w_grad_exp.item()},\n{b_grad_exp.item()}")


Shape of X and y Respectively : (20640, 8), (20640,)
Feature names of the dataset : ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude'],
Traget name ['MedHouseVal']

We need the feature ' AveRooms ' as X 
and
target ' MedHouseVal ' as y

Then we need to compute one forward and backward pass
On Our X and y
Initial Weight and Bias values resprctively : -1.2682,
1.4640
Initial loss: 68.37389373779297
Gradient of w and b are respectively : -97.71773529052734,
-14.979548454284668


## Modification

In [11]:
import torch
from sklearn.datasets import make_regression
import pandas as pd
import numpy as np
torch.manual_seed(42)

X, y = make_regression(n_samples=100,
                       n_features=5,
                       noise=0.1,
                       random_state=42)


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

X = torch.tensor(X, dtype=torch.float32).to(device)
y = torch.tensor(y, dtype=torch.float32).to(device)


w_mod = torch.randn(5,1, dtype=torch.float32, device=device, requires_grad=True)
b_mod = torch.randn(1, dtype=torch.float32, device=device, requires_grad=True)


y_preds_mod = X @ w_mod + b_mod
loss_mod = torch.mean((y_preds_mod - y.view(100,1))**2)
                      
loss_mod.backward()

In [16]:
y_preds_mod.shape, y.shape

(torch.Size([100, 1]), torch.Size([100]))

In [14]:
(w_mod.grad).shape

torch.Size([5, 1])

### Creation

In [21]:
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)

c = a*b
d = torch.sin(c)

L = d*2

L.backward()
print(a.grad, b.grad)



tensor(5.7610) tensor(3.8407)
